# Salud Chilecito - Demo DAO y plataforma

Este notebook funciona como recorrido de presentacion para Base de Datos II. La idea es mostrar el proyecto sin depender de SQL Developer: primero se valida el modelo de datos de demo, despues se revisa el DAO y finalmente se explican los comandos para conectar con Oracle.

**Formas de uso del proyecto:**

- `http://localhost:8000`: plataforma grafica.
- `http://localhost:8000/bot`: plataforma conversacional con Bot IA local.
- `src/dao`: capa DAO para Oracle, siguiendo la linea del proyecto del profesor.
- `sql/`: scripts Oracle con tablespaces, usuarios, esquema, indices, seed y validaciones.


## 1. Preparacion

Ejecutar el notebook desde la raiz del repositorio:

```bash
jupyter notebook notebooks/SaludChilecito_DAO_Demo.ipynb
```

Si todavia no se instalaron dependencias:

```bash
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Repositorio:", ROOT)
print("Existe src/dao:", (ROOT / "src" / "dao").exists())
print("Existe sql/:", (ROOT / "sql").exists())


## 2. Datos de demo

La plataforma web puede funcionar aunque Oracle no este levantado. Para eso usa `data/demo_seed.json` y guarda cambios de prueba en `runtime/salud_chilecito_data.json`.


In [ ]:
seed_path = ROOT / "data" / "demo_seed.json"
data = json.loads(seed_path.read_text(encoding="utf-8"))

for name in ["centros", "especialidades", "medicos", "pacientes", "turnos", "documentos"]:
    print(f"{name}: {len(data[name])}")


In [ ]:
import pandas as pd

pd.DataFrame(data["centros"])


## 3. Uso del store local

Esta parte demuestra operaciones reales sin tocar Oracle. Sirve para la exposicion, para probar la interfaz y para que el profesor vea la funcionalidad sin preparar una base externa.


In [ ]:
from tempfile import TemporaryDirectory
from src.webapp.store import JsonStore

tmp = TemporaryDirectory()
demo_store = JsonStore(
    runtime_path=Path(tmp.name) / "runtime.json",
    seed_path=seed_path,
    uploads_dir=Path(tmp.name) / "uploads",
)

paciente = demo_store.create_patient({
    "dni": "50999888",
    "nombre": "Paciente Notebook",
    "telefono": "3825-111111",
    "distrito": "Chilecito",
    "obra_social": "APOS",
})

turno = demo_store.create_turno({
    "paciente_id": paciente["id"],
    "medico_id": 1,
    "fecha": "2026-06-20",
    "hora": "09:30",
    "estado": "CONFIRMADO",
    "precio": 0,
    "motivo": "Control desde notebook",
})

print("Paciente creado:", paciente)
print("Turno creado:", turno["id"], turno["paciente"]["nombre"], turno["medico"]["nombre"])


## 4. Bot IA local

El bot no llama a servicios externos: interpreta comandos frecuentes y opera el mismo store de la plataforma. Esto permite demostrar una segunda forma de uso, conversacional, sin depender de una API paga.


In [ ]:
from src.webapp.bot_agent import BotAgent

bot = BotAgent(demo_store)
respuesta = bot.handle("listar turnos")
print(respuesta["reply"])


## 5. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicacion no deberia armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO`.


In [ ]:
from src.dao import CentroDAO, PacienteDAO, TurnoDAO

print("DAO disponibles:")
print("-", CentroDAO.__name__)
print("-", PacienteDAO.__name__)
print("-", TurnoDAO.__name__)


In [ ]:
class FakeDB:
    def __init__(self):
        self.calls = []

    def fetch_all(self, sql, params=None):
        self.calls.append(("fetch_all", " ".join(sql.split()), params or {}))
        return []

    def fetch_one(self, sql, params=None):
        self.calls.append(("fetch_one", " ".join(sql.split()), params or {}))
        return {"total": 0}

    def execute(self, sql, params=None):
        self.calls.append(("execute", " ".join(sql.split()), params or {}))
        return 1

fake_db = FakeDB()
CentroDAO(fake_db).listar(distrito="Chilecito")
fake_db.calls[-1]


## 6. Conexion real con Oracle

Para usar Oracle real, primero levantar Docker y cargar scripts:

```bash
docker compose up -d
python scripts/setup_oracle.py
```

O usar los scripts del sistema operativo:

```powershell
scripts\windows\03_cargar_oracle.ps1
```

```bash
bash scripts/ubuntu/03_cargar_oracle.sh
```

Despues se puede ejecutar:

```bash
python -m src.main
```

SQL Developer queda solo como herramienta opcional para mirar tablas, no como paso obligatorio.


## 7. Comandos finales de presentacion

```bash
python scripts/check_requirements.py
pytest -q
python -m src.webapp.server
```

Abrir:

- `http://localhost:8000`
- `http://localhost:8000/bot`
